# Repaso 1: Datos estructurados
### Prueba de conocimientos y deducciones sobre datos reales de Kaggle

**Realizado por:** Joaquín Bermeo

## Propósito de la actividad

Consolidar, en una única actividad integradora, lo estudiado sobre datos estructurados: clasificar variables, inspeccionar la calidad de una tabla, limpiar y transformar columnas, combinar tablas, resumir con agrupaciones y, sobre todo, **deducir conclusiones razonadas** a partir de los resultados. Los datos provienen de un conjunto público de Kaggle (pasajeros del Titanic), por lo que presentan los problemas habituales de los datos reales: valores faltantes, texto sin normalizar y valores extremos.

## Contenidos que se repasan

- Tipos de datos y escalas de medición (notebook de tipos de datos).
- Inspección, selección y filtrado, limpieza, transformación, combinación, agregación y reestructuración (notebook de datos estructurados).
- Interpretación de resultados y razonamiento sobre asociaciones.

## Instrucciones

- Trabaje las secciones en orden: varios ejercicios reutilizan resultados o funciones de ejercicios anteriores.
- Registre cada resultado en la variable `respuesta_N` (donde N es el número del ejercicio), respetando exactamente el formato pedido.
- No modifique las celdas de preparación de datos; trabaje siempre sobre copias de los datos originales.
- Los ejercicios piden **cálculos y deducciones**, no gráficos.
- Los redondeos se indican en cada enunciado. Ante un empate se aplica la regla que se indique; si no se indica ninguna, no hay empates.
- Puntaje total de la prueba: **15 puntos**.

## Preparación de los datos

El conjunto de datos corresponde a los pasajeros del Titanic, publicado en Kaggle. La celda siguiente intenta descargarlo con `kagglehub`; si no es posible, usa una copia pública del mismo conjunto y, como último recurso, un conjunto de respaldo sintético con la misma estructura para que el notebook siempre pueda ejecutarse.

In [2]:
import os, re, io, json, glob, math, warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

In [3]:
COLUMNAS_TITANIC = ["PassengerId", "Survived", "Pclass", "Name", "Sex", "Age", "SibSp", "Parch", "Ticket", "Fare", "Cabin", "Embarked"]
SLUGS_KAGGLE = ["yasserh/titanic-dataset", "hesh97/titanicdataset-traincsv"]
URL_COPIA_PUBLICA = os.environ.get("TITANIC_URL", "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv")


def _es_titanic(d):
    return set(COLUMNAS_TITANIC) <= set(d.columns) and len(d) >= 500


def _respaldo_sintetico(n=891, seed=11):
    rng = np.random.default_rng(seed)
    sexo = rng.choice(["male", "female"], n, p=[.65, .35])
    pclase = rng.choice([1, 2, 3], n, p=[.24, .21, .55])
    t_m = rng.choice(["Mr", "Master", "Dr", "Rev"], n, p=[.90, .07, .015, .015])
    t_f = rng.choice(["Mrs", "Miss", "Ms"], n, p=[.40, .55, .05])
    titulo = np.where(sexo == "male", t_m, t_f)
    ap = ["Smith", "Johnson", "Andersson", "Brown", "Murphy", "Rossi", "Garcia", "Lee"]
    nom = ["John", "Mary", "Anna", "William", "James", "Emma", "Peter", "Lucy"]
    nombres = [f"{ap[int(rng.integers(0, 8))]}, {titulo[i]}. {nom[int(rng.integers(0, 8))]} {nom[int(rng.integers(0, 8))]}" for i in range(n)]
    edad = np.round(np.clip(rng.normal(30, 14, n), 0.5, 80), 1)
    edad[rng.random(n) < .20] = np.nan
    tarifa = np.where(pclase == 1, rng.lognormal(4.2, .6, n), np.where(pclase == 2, rng.lognormal(2.7, .4, n), rng.lognormal(2.1, .5, n)))
    p_sup = np.where(sexo == "female", .72, .17) + np.where(pclase == 1, .12, np.where(pclase == 3, -.06, 0))
    cabina = np.where(rng.random(n) < .23, [f"{rng.choice(list('ABCDE'))}{int(rng.integers(1, 99))}" for _ in range(n)], None)
    emb = rng.choice(["S", "C", "Q"], n, p=[.72, .19, .09]).astype(object)
    emb[rng.choice(n, 2, replace=False)] = None
    return pd.DataFrame({
        "PassengerId": np.arange(1, n + 1), "Survived": (rng.random(n) < p_sup).astype(int), "Pclass": pclase,
        "Name": nombres, "Sex": sexo, "Age": edad, "SibSp": rng.poisson(.5, n), "Parch": rng.poisson(.4, n),
        "Ticket": [str(int(rng.integers(1000, 999999))) for _ in range(n)], "Fare": np.round(tarifa, 4),
        "Cabin": cabina, "Embarked": emb})


def cargar_titanic():
    try:
        import kagglehub
        for slug in SLUGS_KAGGLE:
            try:
                ruta = kagglehub.dataset_download(slug)
                for f in sorted(glob.glob(os.path.join(ruta, "**", "*.csv"), recursive=True)):
                    d = pd.read_csv(f)
                    if _es_titanic(d):
                        return d[COLUMNAS_TITANIC].copy(), f"Kaggle ({slug})"
            except Exception:
                continue
    except Exception:
        pass
    try:
        d = pd.read_csv(URL_COPIA_PUBLICA)
        if _es_titanic(d):
            return d[COLUMNAS_TITANIC].copy(), "copia publica del mismo conjunto de datos (GitHub)"
    except Exception:
        pass
    return _respaldo_sintetico(), "RESPALDO SINTETICO (sin conexion): los resultados seran distintos a los del conjunto real"


dfcpy, FUENTE = cargar_titanic()
print("Fuente de los datos:", FUENTE)
print("Dimensiones:", df.shape)
df.head()

Fuente de los datos: Kaggle (yasserh/titanic-dataset)
Dimensiones: (891, 12)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


El DataFrame `df` contiene los datos originales. **No lo modifique**: en cada ejercicio trabaje sobre una copia (`df.copy()`).

# Ejercicios

## Ejercicio 1. Inspección inicial de la tabla (1 punto)

**Propósito.** Comprobar que sabe realizar la inspección inicial de una tabla (dimensiones y valores faltantes), primer paso de cualquier análisis de datos estructurados.

**Enunciado.** Con el DataFrame `df`, determine:

- el número de filas y de columnas,
- el nombre de la columna con **mayor cantidad de valores nulos** y cuántos nulos tiene.

**Formato de respuesta:** diccionario con las claves `filas` (int), `columnas` (int), `columna_mas_nulos` (str) y `nulos` (int).

In [4]:
# Escriba su solución
RESP = {}
def registrar(n, valor):
    RESP[n] = valor
    print(json.dumps(valor, ensure_ascii = False, indent = 1, default = lambda o : o.item() if hasattr(0, 'item') else str(0)))

dfcpy = df.copy()
nulos = dfcpy.isna().sum()
respuesta_1 = {"filas": int(dfcpy.shape[0]), "columnas": int(dfcpy.shape[1]), "max_nulos" : str(nulos.idxmax()), "nulos" : int(nulos.max())}

registrar(1, respuesta_1)

# Escriba su solución
RESP = {}
def registrar(n,valor):
    RESP[n] = valor
    print(json.dumps(valor, ensure_ascii = False, indent = 1, default = lambda o : o.item()if hasattr(o, 'item')else str(0)))
df = dfcpy.copy()
nulos = df.isna().sum()
respuesta_1 = {"filas": int(df.shape[0]), "columnas": int(df.shape[1]),
              'Max_nulos' : str(nulos.idxmax()),'nulos': int (nulos.max()) }
registrar(1,respuesta_1)
respuesta_1 = None

{
 "filas": 891,
 "columnas": 12,
 "max_nulos": "Cabin",
 "nulos": 687
}


## Ejercicio 2. Clasificación de las variables (2 puntos)

**Propósito.** Evaluar si puede clasificar cada variable según su naturaleza y escala de medición, y no solo según el tipo técnico que muestra pandas (por ejemplo, un número puede ser un identificador o una categoría ordenada).

**Enunciado.** Clasifique cada una de las 12 columnas de `df` usando **exclusivamente** una de estas etiquetas: `identificador`, `binaria`, `nominal`, `ordinal`, `discreta`, `continua`.

**Formato de respuesta:** diccionario `{nombre_de_columna: etiqueta}` con las 12 columnas.

In [5]:
# Solución Ejercicio 2: Clasificación de variables
respuesta_2 = {
    "PassengerId": "identificador",
    "Survived": "binaria",
    "Pclass": "ordinal",
    "Name": "nominal",
    "Sex": "binaria",
    "Age": "continua",
    "SibSp": "discreta",
    "Parch": "discreta",
    "Ticket": "nominal",
    "Fare": "continua",
    "Cabin": "nominal",
    "Embarked": "nominal"
}

registrar(2, respuesta_2)


{
 "PassengerId": "identificador",
 "Survived": "binaria",
 "Pclass": "ordinal",
 "Name": "nominal",
 "Sex": "binaria",
 "Age": "continua",
 "SibSp": "discreta",
 "Parch": "discreta",
 "Ticket": "nominal",
 "Fare": "continua",
 "Cabin": "nominal",
 "Embarked": "nominal"
}


## Ejercicio 3. Filtrado con condiciones combinadas (1 punto)

**Propósito.** Practicar la selección de filas mediante condiciones lógicas combinadas, equivalente al WHERE de SQL.

**Enunciado.** Considere las pasajeras (`Sex` igual a `female`) de primera clase (`Pclass` igual a 1), con edad registrada (`Age` no nula), que sobrevivieron (`Survived` igual a 1).

**Formato de respuesta:** diccionario con `cantidad` (int) y `edad_media` (float, 2 decimales).

In [6]:
condicion = (dfcpy["Sex"] == "female") & (dfcpy["Pclass"] == 1) & (dfcpy["Age"].notna()) & (dfcpy["Survived"] == 1)

respuesta_3 = {"cantidad" : int(condicion.sum()), "edad_media" : round(np.mean(dfcpy[condicion]["Age"]),2)}
registrar(3, respuesta_3)

{
 "cantidad": 82,
 "edad_media": 34.94
}


## Ejercicio 4. Valores faltantes e imputación (2 puntos)

**Propósito.** Evaluar el tratamiento de datos faltantes: medir su magnitud y decidir una estrategia de imputación coherente con la estructura de los datos.

**Enunciado.** Sobre una copia de `df`:

1. Calcule el porcentaje de nulos (1 decimal) de `Age`, `Cabin` y `Embarked` en los datos originales.
2. Impute `Age` con la **mediana de su grupo** definido por `Pclass` y `Sex`.
3. Impute `Embarked` con su **moda**.

**Formato de respuesta:** diccionario con `pct_nulos` (diccionario columna a porcentaje), `edad_media_tras_imputar` (2 decimales), `nulos_age_restantes` (int) y `embarked_imputado_con` (str).

In [7]:
# Escriba su solución
dfcpy = df.copy()
porcentaje_nulos = {c: round(float(dfcpy[c].isna().mean() * 100), 1) for c in ["Age", "Cabin", "Embarked"]}

dfcpy["Age"] = dfcpy["Age"].fillna(dfcpy.groupby(["Pclass","Sex"])["Age"].transform("median"))
moda = df["Embarked"].mode()[0]

respuesta_4 = {"pct_nulos":porcentaje_nulos, "edad media tras imputar":round(float(dfcpy["Age"].mean()),2), "nulos_age_restantes":int(dfcpy["Age"].isna().sum()),"embarked_imputado_con":str(moda)}
registrar(4, respuesta_4)

{
 "pct_nulos": {
  "Age": 19.9,
  "Cabin": 77.1,
  "Embarked": 0.2
 },
 "edad media tras imputar": 29.11,
 "nulos_age_restantes": 0,
 "embarked_imputado_con": "S"
}


## Ejercicio 5. Extracción de información desde texto (1 punto)

**Propósito.** Aplicar operaciones de texto sobre una columna estructurada para derivar una variable nueva, conectando el trabajo con cadenas y las expresiones regulares.

**Enunciado.** En `Name`, el **título** de cada pasajero es el texto comprendido entre la primera coma y el primer punto (por ejemplo, en `Braund, Mr. Owen Harris` el título es `Mr`).

Obtenga la columna de títulos y determine:

- cuántos títulos distintos existen,
- el título más frecuente y su frecuencia,
- cuántos títulos aparecen **una sola vez**.

**Formato de respuesta:** diccionario con `titulos_distintos` (int), `mas_frecuente` (str), `frecuencia` (int) y `titulos_unicos` (int).

In [8]:
# Escriba su solución
dfcpy = df.copy()
titulos = dfcpy["Name"].str.extract(r",\s*([^.]+)\.")[0].str.strip()

conteo = titulos.value_counts()

respuesta_5 = {
    "titulos_distintos": int(titulos.nunique()),
    "mas_frecuente": str(conteo.idxmax()),
    "frecuencia": int(conteo.max()),
    "titulos_unicos": int((conteo == 1).sum())
}
registrar(5, respuesta_5)

{
 "titulos_distintos": 17,
 "mas_frecuente": "Mr",
 "frecuencia": 517,
 "titulos_unicos": 8
}


## Ejercicio 6. Discretización y tasas por grupo (1 punto)

**Propósito.** Practicar la conversión de una variable continua en categorías ordenadas y el resumen por grupos (group by).

**Enunciado.** Cree `grupo_edad` a partir de `Age` con estos intervalos, **abiertos a la izquierda y cerrados a la derecha**: (0, 12] `nino`, (12, 18] `adolescente`, (18, 60] `adulto`, (60, 120] `mayor`. Los pasajeros sin edad se excluyen.

La tasa de supervivencia de un grupo es la media de `Survived`.

**Formato de respuesta:** diccionario con `grupo` (el de mayor tasa, str), `tasa` (3 decimales) y `n` (cantidad de pasajeros de ese grupo).

In [16]:
# Escriba su solución
dfcpy = df.copy()

df_edad = dfcpy.dropna(subset=["Age"]).copy()

intervalos_edad = [0, 12, 18, 60, 120]
etiquetas = ["nino", "adolescente", "adulto", "mayor"]

df_edad["grupo_edad"] = pd.cut(df_edad["Age"], bins=intervalos_edad, labels=etiquetas, right=True)

resumen = df_edad.groupby("grupo_edad", observed=False)["Survived"].agg(tasa="mean", n="count")

grupo_max = resumen["tasa"].idxmax()

respuesta_6 = {
    "grupo": str(grupo_max),
    "tasa": round(float(resumen.loc[grupo_max, "tasa"]), 3),
    "n": int(resumen.loc[grupo_max, "n"])
}

display(df_edad)
display(resumen)
registrar(6, respuesta_6)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,grupo_edad
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S,adulto
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C,adulto
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,adulto
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S,adulto
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S,adulto
...,...,...,...,...,...,...,...,...,...,...,...,...,...
885,886,0,3,"Rice, Mrs. William (Margaret Norton)",female,39.0,0,5,382652,29.1250,NaN,Q,adulto
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,NaN,S,adulto
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,B42,S,adulto
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.0000,C148,C,adulto


,tasa,n
grupo_edad,,
nino,0.579710,69
adolescente,0.428571,70
adulto,0.388788,553
mayor,0.227273,22


{
 "grupo": "nino",
 "tasa": 0.58,
 "n": 69
}


## Ejercicio 7. Combinación de tablas (1 punto)

**Propósito.** Evaluar el uso de una tabla auxiliar (tabla de referencia) y la unión de tablas por una clave, equivalente a un JOIN.

**Enunciado.** Se entrega la tabla auxiliar `puertos`. Combínela con `df` por `Embarked` (ignore los pasajeros sin puerto).

**Formato de respuesta:** diccionario con `por_puerto` (diccionario nombre de puerto a cantidad de pasajeros), `puerto_mayor` (str) y `porcentaje_mayor` (porcentaje que representa sobre los pasajeros con puerto conocido, 1 decimal).

In [10]:
puertos = pd.DataFrame({"Embarked": ["C", "Q", "S"], "Puerto": ["Cherbourg", "Queenstown", "Southampton"]})
puertos

,Embarked,Puerto
0,C,Cherbourg
1,Q,Queenstown
2,S,Southampton


In [19]:
# Escriba su solución
dfcpy = df.copy()

df_combinado = dfcpy.merge(puertos, on="Embarked", how="inner")

conteo_puertos = df_combinado["Puerto"].value_counts()

puerto_mayor = str(conteo_puertos.idxmax())
total_conocidos = len(df_combinado)
porcentaje_mayor = round(float((conteo_puertos.max() / total_conocidos) * 100), 2)

por_puerto = conteo_puertos.to_dict() # to_dict transforma dirrectamente una Serie en un Diccionario.

respuesta_7 = {
    "por_puerto": por_puerto,
    "puerto_mayor": puerto_mayor,
    "porcentaje_mayor": porcentaje_mayor
}
registrar(7, respuesta_7)

{
 "por_puerto": {
  "Southampton": 644,
  "Cherbourg": 168,
  "Queenstown": 77
 },
 "puerto_mayor": "Southampton",
 "porcentaje_mayor": 72.44
}


## Ejercicio 8. Tabla dinámica de dos factores (1 punto)

**Propósito.** Practicar la reestructuración de datos con una tabla dinámica y la lectura de extremos en un cruce de dos variables categóricas.

**Enunciado.** Construya la tasa de supervivencia (3 decimales) para cada combinación de `Sex` y `Pclass` e identifique las combinaciones con la **menor** y la **mayor** tasa.

**Formato de respuesta:** diccionario con las claves `menor` y `mayor`, cada una con `Sex` (str), `Pclass` (int) y `tasa` (float).

In [22]:
# Escriba su solución
dfcpy = df.copy()

tasas = dfcpy.groupby(["Sex", "Pclass"])["Survived"].mean()
tasa_menor = tasas.idxmin()
tasa_mayor = tasas.idxmax()

display(tasas)
respuesta_8 = {
    "menor": {
        "Sex": str(tasa_menor[0]),
        "Pclass": int(tasa_menor[1]),
        "tasa": round(float(tasas[tasa_menor]), 3)
    },
    "mayor": {
        "Sex": str(tasa_mayor[0]),
        "Pclass": int(tasa_mayor[1]),
        "tasa": round(float(tasas[tasa_mayor]), 3)
    }
}
registrar(8, respuesta_8)

Sex     Pclass
female  1         0.968085
        2         0.921053
        3         0.500000
male    1         0.368852
        2         0.157407
        3         0.135447
Name: Survived, dtype: float64

{
 "menor": {
  "Sex": "male",
  "Pclass": 3,
  "tasa": 0.135
 },
 "mayor": {
  "Sex": "female",
  "Pclass": 1,
  "tasa": 0.968
 }
}


## Ejercicio 9. Variable derivada y comparación (1 punto)

**Propósito.** Evaluar la creación de una variable derivada a partir de varias columnas y la comparación de tasas entre subgrupos.

**Enunciado.** Cree `tam_familia` como `SibSp + Parch + 1`. Un pasajero **viaja solo** si `tam_familia` es 1.

Calcule la tasa de supervivencia (3 decimales) de quienes viajan solos y de quienes viajan acompañados, y además el `tam_familia` con **mayor tasa de supervivencia** considerando únicamente tamaños con al menos 10 pasajeros.

**Formato de respuesta:** diccionario con `tasa_solo`, `tasa_acompanado`, `tam_mayor_tasa_min10` (int) y `tasa_tam` (3 decimales).

In [23]:
# Escriba su solución
dfcpy = df.copy()

dfcpy["tam_familia"] = dfcpy["SibSp"] + dfcpy["Parch"] + 1

tasa_solo = round(float(dfcpy[dfcpy["tam_familia"] == 1]["Survived"].mean()), 3)
tasa_acompanado = round(float(dfcpy[dfcpy["tam_familia"] > 1]["Survived"].mean()), 3)

resumen_familia = dfcpy.groupby("tam_familia")["Survived"].agg(tasa="mean", n="count")

resumen_filtrado = resumen_familia[resumen_familia["n"] >= 10]

tam_mayor = resumen_filtrado["tasa"].idxmax()
tasa_tam = round(float(resumen_filtrado.loc[tam_mayor, "tasa"]), 3)

respuesta_9 = {
    "tasa_solo": tasa_solo,
    "tasa_acompanado": tasa_acompanado,
    "tam_mayor_tasa_min10": int(tam_mayor),
    "tasa_tam": tasa_tam
}
registrar(9, respuesta_9)

{
 "tasa_solo": 0.304,
 "tasa_acompanado": 0.506,
 "tam_mayor_tasa_min10": 4,
 "tasa_tam": 0.724
}


## Ejercicio 10. Valores atípicos con la regla del IQR (1 punto)

**Propósito.** Evaluar la detección de valores atípicos con un criterio estadístico estándar y su cuantificación.

**Enunciado.** Para `Fare`, calcule el rango intercuartílico (IQR) con los cuantiles 0.25 y 0.75 y considere atípico todo valor **mayor** que `Q3 + 1.5 x IQR`.

**Formato de respuesta:** diccionario con `limite_superior` (2 decimales), `n_atipicos` (int), `pct_atipicos` (porcentaje sobre todas las filas, 1 decimal) y `maximo` (2 decimales).

In [24]:
# Escriba su solución
dfcpy = df.copy()

q1 = dfcpy["Fare"].quantile(0.25)
q3 = dfcpy["Fare"].quantile(0.75)
iqr = q3 - q1

limite_superior = q3 + 1.5 * iqr

atipicos = dfcpy[dfcpy["Fare"] > limite_superior]
n_atipicos = len(atipicos)
pct_atipicos = round((n_atipicos / len(dfcpy)) * 100, 1)
maximo = round(float(dfcpy["Fare"].max()), 2)

respuesta_10 = {
    "limite_superior": round(float(limite_superior), 2),
    "n_atipicos": int(n_atipicos),
    "pct_atipicos": float(pct_atipicos),
    "maximo": maximo
}

registrar(10, respuesta_10)

{
 "limite_superior": 65.63,
 "n_atipicos": 116,
 "pct_atipicos": 13.0,
 "maximo": 512.33
}


## Ejercicio 11. Deducción razonada (3 puntos)

**Propósito.** Evaluar la capacidad de pasar de los números a conclusiones: interpretar resultados, comparar grupos y reconocer los límites de lo que los datos permiten afirmar.

**Enunciado.** Con base en los resultados de los ejercicios 6, 8 y 9, redacte entre 5 y 8 líneas que expliquen **qué factores se asocian con la supervivencia** en estos datos. Incluya al menos **tres cifras** obtenidas en sus cálculos y una frase que explique **por qué estos resultados no permiten afirmar causalidad**.

**Formato de respuesta:** texto redactado en la celda de respuesta.

**Escriba aquí su respuesta:**

Los datos revelan que la probabilidad de sobrevivir dependió en gran medida de la edad, el sexo, la clase del boleto y la compañía. Se dio prioridad alta a los más vulnerables, es decir, niños y mujeres. Los niños alcanzaron una supervivencia del 58.0% frente al 22.7% de los adultos mayores, y la mayor tasa la tuvieron las mujeres de primera clase con un 96.8%, mientras que los hombres de tercera clase apenas llegaron al 13.5%. Por otro lado, viajar acompañado ayudó a que la tasa de supervivencia sea mayor, quienes iban con familia registraron un 50.6% de supervivencia (frente al 30.4% de quienes viajaban solos), llegando a un máximo del 72.4% en grupos de cuatro personas. Sin embargo, estos datos solo muestran asociaciones y no permiten confirmar si fueron coincidencias o no, ya que no se trató de un experimento controlado y situaciones como la ubicación de la habitación pudieron haber afectado el resultado.